In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('tpch_dataset.csv')
print(df.shape)
df.head()

(2100, 7)


,machine_id,query_id,cost,time_ms,shared_hit,shared_read,rows
0,c5a,q10,160189.8,536.241,154869,104275,20
1,c5a,q10,160189.8,534.430,155138,104006,20
2,c5a,q10,160189.8,530.821,155124,104020,20
3,c5a,q10,160189.8,528.013,154920,104224,20
4,c5a,q10,160189.8,534.745,154991,104153,20


In [ ]:
agg = df.groupby(['machine_id', 'query_id']).agg(
    time_p50=('time_ms', lambda x: np.percentile(x, 50)),
    time_p95=('time_ms', lambda x: np.percentile(x, 95)),
    time_p99=('time_ms', lambda x: np.percentile(x, 99)),
    time_cv=('time_ms', lambda x: 100 * np.std(x) / np.mean(x)),
    cost=('cost', 'first'),
    shared_hit=('shared_hit', 'median'),
    shared_read=('shared_read', 'median'),
    rows=('rows', 'median'),
).reset_index()

agg['total_buffers'] = agg['shared_hit'] + agg['shared_read']
agg['io_ratio'] = agg['shared_read'] / (agg['total_buffers'] + 1)
print(agg.shape)
agg.head()

(105, 12)


,machine_id,query_id,time_p50,time_p95,time_p99,time_cv,cost,shared_hit,shared_read,rows,total_buffers,io_ratio
0,c5a,q1,3043.2095,3237.00370,5554.07194,21.019673,231253.32,926.0,111688.0,4.0,112614.0,0.991768
1,c5a,q10,530.1745,536.26705,536.66301,1.476125,160189.80,154977.0,104167.0,20.0,259144.0,0.401964
2,c5a,q11,85.4290,91.53775,140.91875,16.641400,10203.71,62785.0,0.0,1217.0,62785.0,0.000000
3,c5a,q12,717.6430,725.08935,750.41787,1.277855,210511.91,1780.0,137090.0,2.0,138870.0,0.987175
4,c5a,q13,833.7245,851.47365,853.64193,1.307681,142161.33,6288.0,23465.0,42.0,29753.0,0.788633


In [ ]:
machines = agg['machine_id'].unique()
pairs = []

for dev in machines:
    for prod in machines:
        if dev == prod:
            continue
        dev_data = agg[agg['machine_id'] == dev].set_index('query_id')
        prod_data = agg[agg['machine_id'] == prod].set_index('query_id')
        common_queries = dev_data.index.intersection(prod_data.index)

        for q in common_queries:
            pairs.append({
                'dev_machine': dev,
                'prod_machine': prod,
                'query_id': q,
                'dev_cost': dev_data.loc[q, 'cost'],
                'dev_total_buffers': dev_data.loc[q, 'total_buffers'],
                'dev_io_ratio': dev_data.loc[q, 'io_ratio'],
                'dev_time_p50': dev_data.loc[q, 'time_p50'],
                'prod_time_p50': prod_data.loc[q, 'time_p50'],
                'prod_time_p95': prod_data.loc[q, 'time_p95'],
                'prod_time_p99': prod_data.loc[q, 'time_p99'],
                'scaling_factor_p50': prod_data.loc[q, 'time_p50'] / dev_data.loc[q, 'time_p50'],
            })

pairs_df = pd.DataFrame(pairs)
print(pairs_df.shape)
pairs_df.head()

(420, 11)


,dev_machine,prod_machine,query_id,dev_cost,dev_total_buffers,dev_io_ratio,dev_time_p50,prod_time_p50,prod_time_p95,prod_time_p99,scaling_factor_p50
0,c5a,c7i,q1,231253.32,112614.0,0.991768,3043.2095,4861.0350,4958.10595,5909.74679,1.597338
1,c5a,c7i,q10,160189.80,259144.0,0.401964,530.1745,844.8750,864.14180,875.66036,1.593579
2,c5a,c7i,q11,10203.71,62785.0,0.000000,85.4290,102.7545,113.66250,155.94130,1.202806
3,c5a,c7i,q12,210511.91,138870.0,0.987175,717.6430,1038.8170,1050.74075,1055.49455,1.447540
4,c5a,c7i,q13,142161.33,29753.0,0.788633,833.7245,1499.6860,1515.10705,1518.07181,1.798779


In [ ]:
print("Total pairs:", len(pairs_df))
print("Expected:", 5*4*21, "(5 machines, 4 targets each, 21 queries)")
print()
print("Scaling factor distribution:")
print(pairs_df['scaling_factor_p50'].describe())
print()
print("Correlation: dev_cost vs scaling_factor:", pairs_df['dev_cost'].corr(pairs_df['scaling_factor_p50']))
print("Correlation: dev_total_buffers vs scaling_factor:", pairs_df['dev_total_buffers'].corr(pairs_df['scaling_factor_p50']))

Total pairs: 420
Expected: 420 (5 machines, 4 targets each, 21 queries)

Scaling factor distribution:
count    420.000000
mean       1.069828
std        0.399138
min        0.422971
25%        0.766944
50%        1.000004
75%        1.303882
max        2.364227
Name: scaling_factor_p50, dtype: float64

Correlation: dev_cost vs scaling_factor: 0.01070446563034863
Correlation: dev_total_buffers vs scaling_factor: -0.005497524188817438


In [ ]:
pairs_df['pair_id'] = pairs_df['dev_machine'] + '_to_' + pairs_df['prod_machine']
print(pairs_df.groupby('pair_id')['scaling_factor_p50'].agg(['mean','std','count']))

                mean       std  count
pair_id                              
c5a_to_c7i  1.455223  0.347881     21
c5a_to_m5a  1.480787  0.065015     21
c5a_to_r5n  1.842526  0.409232     21
c5a_to_z1d  1.533310  0.386648     21
c7i_to_c5a  0.728672  0.190571     21
c7i_to_m5a  1.084180  0.319705     21
c7i_to_r5n  1.274623  0.135838     21
c7i_to_z1d  1.055117  0.127005     21
m5a_to_c5a  0.676531  0.029064     21
m5a_to_c7i  0.986864  0.242274     21
m5a_to_r5n  1.251415  0.293797     21
m5a_to_z1d  1.042264  0.276380     21
r5n_to_c5a  0.575650  0.158846     21
r5n_to_c7i  0.793401  0.087910     21
r5n_to_m5a  0.858091  0.270284     21
r5n_to_z1d  0.827507  0.053242     21
z1d_to_c5a  0.703239  0.219884     21
z1d_to_c7i  0.964058  0.144204     21
z1d_to_m5a  1.049668  0.370812     21
z1d_to_r5n  1.213436  0.081715     21


In [ ]:
sig = {
    'c5a': {'bandwidth': 14.9, 'compute': 0.593},
    'z1d': {'bandwidth': 9.77, 'compute': 0.882},
    'r5n': {'bandwidth': 9.39, 'compute': 0.705},
    'm5a': {'bandwidth': 10.01, 'compute': 0.457},
    'c7i': {'bandwidth': 8.44, 'compute': 0.807},
}

pairs_df['dev_bandwidth'] = pairs_df['dev_machine'].map(lambda m: sig[m]['bandwidth'])
pairs_df['prod_bandwidth'] = pairs_df['prod_machine'].map(lambda m: sig[m]['bandwidth'])
pairs_df['dev_compute'] = pairs_df['dev_machine'].map(lambda m: sig[m]['compute'])
pairs_df['prod_compute'] = pairs_df['prod_machine'].map(lambda m: sig[m]['compute'])

pairs_df['bandwidth_ratio'] = pairs_df['prod_bandwidth'] / pairs_df['dev_bandwidth']
pairs_df['compute_ratio'] = pairs_df['prod_compute'] / pairs_df['dev_compute']

print(pairs_df['bandwidth_ratio'].corr(pairs_df['scaling_factor_p50']))
print(pairs_df['compute_ratio'].corr(pairs_df['scaling_factor_p50']))

-0.6384661227794786
0.1683720796308118
